In [7]:
import pandas as pd
import numpy as np

# 📡 Kapsama hesapları için
from coverage import calculate_coverage  
from path_config import PathConfig

# 📊 Performans metrikleri için
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [8]:
def run_coverage_evaluation(
    tx_lat,
    tx_lon,
    tx_height_m,
    target_column,
    azimuth_deg=0,
    h_bw_deg=65,
    v_bw_deg=10,
    sheet_name="Series Formatted Data",
    pci_target=30,
):
    paths = PathConfig()
    dl_path = paths.dl_path

    # 2. Veriyi oku
    df = pd.read_excel(dl_path, sheet_name=sheet_name)

    # 3. Filtreleme
    df_target = df[df["NR_UE_PCI_0"] == pci_target]
    df_target = df_target[df_target[target_column].notna()]
    # Her bir satır için coverage hesapla ve sonuçları birleştir
    results = []
    for _, row in df_target.iterrows():
        coverage_result = calculate_coverage(
            tx_lat=tx_lat,
            tx_lon=tx_lon,
            tx_height_m=tx_height_m,
            rx_lat=row["Latitude"],
            rx_lon=row["Longitude"],
            rx_height_m=0,
            azimuth_deg=azimuth_deg,
            h_bw_deg=h_bw_deg,
            v_bw_deg=v_bw_deg
        )
        try:
            pred_value = float(coverage_result[target_column].iloc[0])
        except Exception as e:
            print(f"Hata: {e}, coverage_result: {coverage_result}, target_column: {target_column}")
            pred_value = np.nan
        results.append({
            "Latitude": row["Latitude"],
            "Longitude": row["Longitude"],
            "Time": row["Time"],
            "Predicted Value": pred_value,
            "Actual Value": row[target_column]
        })

    coverage_df = pd.DataFrame(results)

    # Sklearn metriklerini hesapla ve yazdır
    if not coverage_df.empty:
        y_true = coverage_df["Actual Value"]
        y_pred = coverage_df["Predicted Value"]
        print("MAE:", mean_absolute_error(y_true, y_pred))
        print("MSE:", mean_squared_error(y_true, y_pred))
        print("RMSE:", mean_squared_error(y_true, y_pred, squared=False))
        print("R2:", r2_score(y_true, y_pred))

    return coverage_df

In [9]:
if __name__ == "__main__":
    # Değişkenleri elle giriyoruz
    tx_lat = 41.105096
    tx_lon = 29.025006
    tx_height_m = 30
    azimuth_deg = 0
    h_bw_deg = 65
    v_bw_deg = 10
    pci_target = 30
    sheet_name = "Series Formatted Data"

    target_columns = [
        "NR_UE_RSRP_0",
        "NR_UE_Timing_Advance",
        "NR_UE_Pathloss_DL_0"
    ]

    for target_col in target_columns:
        print(f"\n--- {target_col} için sonuçlar ---")
        df_result = run_coverage_evaluation(
            tx_lat=tx_lat,
            tx_lon=tx_lon,
            tx_height_m=tx_height_m,
            azimuth_deg=azimuth_deg,
            h_bw_deg=h_bw_deg,
            v_bw_deg=v_bw_deg,
            pci_target=pci_target,
            sheet_name=sheet_name,
            target_column=target_col
        )
        print(df_result.head())


--- NR_UE_RSRP_0 için sonuçlar ---
MAE: 99.80193050193049
MSE: 12592.39808030888
RMSE: 112.2158548526405
R2: -72.0099119126429
   Latitude  Longitude                    Time  Predicted Value  Actual Value
0  41.10760   29.02257 2025-03-14 12:18:50.499          -206.03         -81.6
1  41.10757   29.02263 2025-03-14 12:18:51.014          -205.75         -81.4
2  41.10756   29.02266 2025-03-14 12:18:51.587          -205.63         -84.3
3  41.10755   29.02268 2025-03-14 12:18:52.016          -205.53         -85.9
4  41.10755   29.02268 2025-03-14 12:18:52.499          -205.53         -84.1

--- NR_UE_Timing_Advance için sonuçlar ---


/Users/2na/miniconda3/lib/python3.11/site-packages/sklearn/metrics/_regression.py:483: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


Empty DataFrame
Columns: []
Index: []

--- NR_UE_Pathloss_DL_0 için sonuçlar ---
Empty DataFrame
Columns: []
Index: []


In [10]:
paths = PathConfig()
dl_path = paths.dl_path
df = pd.read_excel(dl_path, sheet_name="Series Formatted Data")
df_target = df[df["NR_UE_Timing_Advance"].notna()]
df_target = df_target[df_target["NR_UE_PCI_0"] == 30]

print("\nHedef PCI 30 olan ve Pathloss DL değeri olan ilk 5 satır:")
print(df_target.head())




Hedef PCI 30 olan ve Pathloss DL değeri olan ilk 5 satır:
Empty DataFrame
Columns: [Message, Time, Longitude, Latitude, Technology_Mode, NR_UE_PCI_0, NR_UE_RSRP_0, NR_UE_RSRQ_0, NR_UE_SINR_0, NR_UE_Nbr_PCI_0, NR_UE_Nbr_PCI_1, NR_UE_Nbr_PCI_2, NR_UE_Nbr_PCI_3, NR_UE_Nbr_PCI_4, NR_UE_Nbr_RSRP_0, NR_UE_Nbr_RSRP_1, NR_UE_Nbr_RSRP_2, NR_UE_Nbr_RSRP_3, NR_UE_Nbr_RSRP_4, NR_UE_Nbr_RSRQ_0, NR_UE_Nbr_RSRQ_1, NR_UE_Nbr_RSRQ_2, NR_UE_Nbr_RSRQ_3, NR_UE_Nbr_RSRQ_4, NR_UE_Timing_Advance, NR_UE_Pathloss_DL_0, NR_UE_Throughput_PDCP_DL, App_Throughput_DL, NR_UE_NACK_Rate_DL_0, NR_UE_Ack_As_Nack_DL_0, NR_UE_MCS_DL_0, NR_UE_RB_Num_DL_0, NR_UE_Modulation_Avg_DL_0, NR_UE_RI_DL_0, NR_UE_BLER_DL_0, NR_UE_CCE_AggregationLev_0, NR_UE_Power_Tx_PUSCH_0, NR_UE_Power_Tx_PRACH_0, NR_UE_NACK_Rate_UL_0, NR_UE_RACH_Attempt, NR_UE_RACH_OK, NR_UE_RACH_Fail, NR_UE_RACH_Procedure_Count, NR_UE_RRCReEstAttempt, NR_UE_RRCReEstFail, NR_UE_RRCReEst_EndResult, NR_UE_RRCConnectionAttempt, NR_UE_RRCConnectionSetupOk, Unnamed: 48